## Advanced Bucket Access

This notebook inspects the live bucket catalog and then fetches metadata plus a single chunk from one product.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from maya4 import DEFAULT_BUCKET_ID, list_base_files_in_bucket, parse_product_filename

product_files = sorted(
    file_name
    for file_name in list_base_files_in_bucket(DEFAULT_BUCKET_ID, relative_path=True)
    if file_name.endswith('.zarr')
)

rows = []
for file_name in product_files:
    meta = parse_product_filename(file_name)
    rows.append(
        {
            'filename': file_name,
            'satellite': meta['satellite'].upper(),
            'year': meta['acquisition_date'].year,
            'stripmap_mode': meta['stripmap_mode'],
            'polarization': meta['polarization'].upper(),
        }
    )

catalog = pd.DataFrame(rows).sort_values(['stripmap_mode', 'filename']).reset_index(drop=True)
print('products in bucket:', len(catalog))
print(catalog.head(10).to_string(index=False))
print('\nstripmap counts:')
print(catalog['stripmap_mode'].value_counts().sort_index().to_string())


products in bucket: 32
                                                          filename satellite  year  stripmap_mode polarization
s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr       S1C  2025              1           VV
s1c-s1-raw-s-vv-20250417t025744-20250417t025817-001927-003c59.zarr       S1C  2025              1           VV
s1c-s1-raw-s-vv-20250417t025809-20250417t025828-001927-003c59.zarr       S1C  2025              1           VV
s1c-s1-raw-s-vv-20250424t170801-20250424t170839-002038-0042df.zarr       S1C  2025              1           VV
s1c-s2-raw-s-vv-20250331t205042-20250331t205107-001690-002d5c.zarr       S1C  2025              2           VV
s1c-s2-raw-s-vv-20250403t062508-20250403t062540-001725-002fa3.zarr       S1C  2025              2           VV
s1c-s2-raw-s-vv-20250405t060506-20250405t060539-001754-00317b.zarr       S1C  2025              2           VV
s1c-s2-raw-s-vv-20250405t060531-20250405t060604-001754-00317b.zarr       S1C  2025       

In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import zarr
from maya4 import DEFAULT_BUCKET_ID, download_metadata_from_product, fetch_chunk_from_bucket_zarr, list_base_files_in_bucket

DATA_DIR = REPO_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
product = next(
    file_name
    for file_name in sorted(list_base_files_in_bucket(DEFAULT_BUCKET_ID, relative_path=True))
    if file_name.endswith('.zarr')
)

_ = download_metadata_from_product(
    zfile_name=product,
    local_dir=DATA_DIR,
    bucket_id=DEFAULT_BUCKET_ID,
    levels=['rcmc', 'az'],
    show_progress=False,
)
fetch_chunk_from_bucket_zarr(
    level='rcmc',
    y=0,
    x=0,
    local_dir=DATA_DIR,
    bucket_id=DEFAULT_BUCKET_ID,
    zarr_archive=product,
    show_progress=False,
)
fetch_chunk_from_bucket_zarr(
    level='az',
    y=0,
    x=0,
    local_dir=DATA_DIR,
    bucket_id=DEFAULT_BUCKET_ID,
    zarr_archive=product,
    show_progress=False,
)

rcmc = zarr.open(DATA_DIR / product / 'rcmc', mode='r')
az = zarr.open(DATA_DIR / product / 'az', mode='r')

print('product:', product)
print('local_cache:', (DATA_DIR / product).resolve())
print('rcmc shape:', rcmc.shape, 'chunks:', rcmc.chunks)
print('az shape:', az.shape, 'chunks:', az.chunks)
print('rcmc[0, 0]:', complex(rcmc[0, 0]))
print('az[0, 0]:', complex(az[0, 0]))


product: s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr
local_cache: /Users/roberto.delprete/Downloads/Maya4/data/s1c-s1-raw-s-vv-20250328t052810-20250328t052835-001637-002a0d.zarr
rcmc shape: (44904, 25484) chunks: (256, 256)
az shape: (44904, 25484) chunks: (256, 256)
rcmc[0, 0]: (5112.9243126413685-17.965251846144064j)
az[0, 0]: (2701.2872411565663+417.5754265061424j)
